<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B07%5D%20-%20Ingenieria_de_Variables_I/%5B01%5D%20-%20Notebooks/E2_Codifica_como_un_Pro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E2 · Codifica como un pro - Ingenieria de Variables I

## Introduccion

El ordenador no entiende "España" ni "Amazon": tenemos que **convertir etiquetas en numeros**
sin inventar relaciones falsas. En este ejercicio:

1. Vemos el **error a evitar** (asignar 1, 2, 3 a categorias sin orden).
2. Aplicamos **One-Hot encoding** a las categoricas de baja cardinalidad (`tipo_tarjeta`, `pais`, `canal`).
3. Hablamos del **encoding ordinal** (solo cuando hay orden real).
4. Atacamos la **alta cardinalidad** (`comercio`, con decenas/cientos de categorias).
5. Usamos **Target / Mean encoding** y entendemos su peligro: la **fuga de datos (leakage)**.
6. Target encoding **por turnos (out-of-fold) + smoothing**.
7. Comprobamos si **sube o baja el acierto en datos nuevos**.

## Objetivos del ejercicio

- Codificar categoricas de baja cardinalidad con **One-Hot** correctamente.
- Reconocer cuando es valido el **encoding ordinal**.
- Entender el problema de la **alta cardinalidad** y el **overfitting** del target encoding ingenuo.
- Implementar **target encoding out-of-fold con smoothing** sin fugas.
- Comparar el rendimiento **en datos nuevos** (AUC en test).

## Descripcion del dataset (fraude con tarjeta)

Trabajamos con un dataset **sintetico y reproducible** de transacciones con tarjeta.
Lo generamos dentro del propio notebook para que sea autocontenido en Colab.
Cada fila es una transaccion con estas variables:

| Variable | Tipo | Descripcion |
|---|---|---|
| `id_cliente` | id | Identificador del cliente |
| `edad` | numerica | Edad del cliente (con algunos huecos) |
| `monto` | numerica | Importe de la transaccion en euros (distribucion sesgada) |
| `pais` | categorica | Pais de la operacion: ES, DE, UK, FR, IT |
| `tipo_tarjeta` | categorica | debito / credito / prepago |
| `comercio` | categorica (ALTA cardinalidad) | Comercio donde se opera (COM_0000 ... COM_0299, con frecuencias muy desiguales) |
| `canal` | categorica | online / presencial (con algunos huecos) |
| `codigo_postal` | pseudo-numerica | Parece numero, pero NO tiene magnitud |
| `fecha` | fecha/hora | Momento de la transaccion |
| `es_fraude` | objetivo (0/1) | 1 si la transaccion fue fraudulenta |

> La probabilidad real de fraude se ha construido en funcion del monto, la hora,
> el canal, el pais y el comercio. Por eso, **las variables que creemos tendran
> señal de verdad** y veremos su efecto en las metricas.

### 1. Importar librerias necesarias

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction import FeatureHasher
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_columns", 50)

### 2. Datos y particion train/test

In [ ]:
import numpy as np
import pandas as pd

def generar_datos_fraude(n=8000, semilla=42):
    # Dataset sintetico y REPRODUCIBLE de transacciones con tarjeta.
    # La probabilidad de fraude depende de variables reales (monto, hora,
    # canal, pais y comercio): asi la ingenieria de variables tiene señal de verdad.
    rng = np.random.default_rng(semilla)

    # --- Perfil de clientes ---
    n_clientes = 600
    gasto_medio_cliente = rng.lognormal(mean=3.2, sigma=0.5, size=n_clientes)
    edad_por_cliente = rng.integers(18, 80, size=n_clientes)
    id_cliente = rng.integers(0, n_clientes, size=n)

    # --- Comercios (ALTA cardinalidad, con frecuencias MUY desiguales) ---
    n_comercios = 300
    comercios = np.array([f"COM_{i:04d}" for i in range(n_comercios)])
    riesgo_comercio = rng.beta(1.2, 8.0, size=n_comercios)   # casi todos bajos, unos pocos altos
    peso_comercio = 1.0 / np.arange(1, n_comercios + 1)      # ley de potencias: muchos comercios raros
    peso_comercio = peso_comercio / peso_comercio.sum()
    idx_comercio = rng.choice(n_comercios, size=n, p=peso_comercio)

    # --- Categoricas de BAJA cardinalidad ---
    pais = rng.choice(["ES", "DE", "UK", "FR", "IT"], size=n, p=[0.60, 0.12, 0.10, 0.10, 0.08])
    tipo_tarjeta = rng.choice(["debito", "credito", "prepago"], size=n, p=[0.55, 0.40, 0.05])
    canal = rng.choice(["online", "presencial"], size=n, p=[0.45, 0.55])

    # --- Codigo postal (PSEUDO-numerica: parece numero, pero es categorica) ---
    cp_base = rng.choice([28001, 8001, 41001, 46001, 50001], size=n)
    codigo_postal = cp_base + rng.integers(0, 40, size=n)

    # --- Monto (distribucion sesgada con outliers) ---
    monto = rng.lognormal(mean=np.log(gasto_medio_cliente[id_cliente]), sigma=0.8)
    gigantes = rng.random(n) < 0.005                         # unas pocas compras enormes
    monto[gigantes] *= rng.uniform(20, 80, size=int(gigantes.sum()))
    monto = np.round(monto, 2)

    # --- Fecha y hora ---
    inicio = np.datetime64("2024-01-01T00:00")
    minutos = rng.integers(0, 365 * 24 * 60, size=n)
    fecha = pd.to_datetime(inicio + minutos.astype("timedelta64[m]"))
    hora = fecha.hour.to_numpy()
    dia_semana = fecha.dayofweek.to_numpy()

    # --- Probabilidad de fraude: la señal vive en estas variables ---
    logit = (
        -4.2
        + 0.45 * (np.log1p(monto) - np.log1p(monto).mean())
        + 1.8 * (hora < 6)
        + 0.7 * (canal == "online")
        + 0.5 * (pais != "ES")
        + 4.0 * riesgo_comercio[idx_comercio]
        + 0.3 * (dia_semana >= 5)
    )
    prob = 1.0 / (1.0 + np.exp(-logit))
    es_fraude = rng.binomial(1, prob)

    df = pd.DataFrame({
        "id_cliente": id_cliente,
        "edad": edad_por_cliente[id_cliente].astype(float),
        "monto": monto,
        "pais": pais,
        "tipo_tarjeta": tipo_tarjeta,
        "comercio": comercios[idx_comercio],
        "canal": canal,
        "codigo_postal": codigo_postal,
        "fecha": fecha,
        "es_fraude": es_fraude,
    })

    # Valores faltantes realistas (para practicar imputacion)
    df.loc[rng.random(n) < 0.05, "edad"] = np.nan
    df.loc[rng.random(n) < 0.03, "canal"] = np.nan
    return df

In [ ]:
df = generar_datos_fraude(n=8000, semilla=42)

# SIEMPRE separamos antes de codificar con informacion del target (evita fugas).
train, test = train_test_split(df, test_size=0.3, random_state=0, stratify=df["es_fraude"])
print("train:", train.shape, "| test:", test.shape)
print("Tasa de fraude train:", round(train["es_fraude"].mean(), 4),
      "| test:", round(test["es_fraude"].mean(), 4))

### 3. El error a evitar: etiquetar con numeros

Si ponemos `ES=1, DE=2, UK=3`, el modelo cree que `UK > ES` y que `DE` esta justo en medio.
Inventamos un **orden y unas distancias que no existen**. Para pais o comercio: **nunca**.

In [ ]:
# Demostracion del error (NO hacer esto con categoricas sin orden):
mapa_malo = {"ES": 1, "DE": 2, "UK": 3, "FR": 4, "IT": 5}
ejemplo = train["pais"].map(mapa_malo).head(6)
print("pais -> numero (mal):")
print(pd.DataFrame({"pais": train["pais"].head(6).values, "codigo_malo": ejemplo.values}))
print("\nEl modelo interpretaria IT(5) > ES(1), lo cual no tiene sentido.")

### 4. One-Hot encoding (baja cardinalidad)

Creamos una columna 0/1 por categoria: `pais_ES`, `pais_DE`, … Cada fila marca un 1 en su valor.
Usamos `handle_unknown="ignore"` para que las categorias nuevas en test no rompan nada.

In [ ]:
cols_ohe = ["tipo_tarjeta", "pais", "canal"]
train_ohe_in = train[cols_ohe].fillna("desconocido")
test_ohe_in = test[cols_ohe].fillna("desconocido")

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_ohe = ohe.fit_transform(train_ohe_in)     # fit SOLO con train
test_ohe = ohe.transform(test_ohe_in)

ohe_cols = ohe.get_feature_names_out(cols_ohe)
print("Columnas generadas por One-Hot:", list(ohe_cols))
pd.DataFrame(train_ohe, columns=ohe_cols, index=train.index).head()

### 5. Encoding ordinal: solo con orden real

El encoding ordinal (1, 2, 3) **si** vale cuando hay un orden natural: talla `S < M < L`,
o nivel `bajo < medio < alto`. La clave es que el orden **exista de verdad**.

In [ ]:
# Ejemplo ilustrativo con una variable ordinal de juguete:
tallas = pd.Series(["M", "S", "L", "M", "L", "S"])
orden = {"S": 0, "M": 1, "L": 2}
print(pd.DataFrame({"talla": tallas, "ordinal": tallas.map(orden)}))
print("\nAqui SI tiene sentido: S < M < L es un orden real.")

### 6. El problema de la alta cardinalidad

`comercio` tiene muchas categorias. Con One-Hot tendriamos una columna por comercio:
tablas enormes, lentas y con mucho ruido. Necesitamos algo mas listo.

In [ ]:
print("Nº de comercios distintos:", train["comercio"].nunique())
print("Con One-Hot crearia", train["comercio"].nunique(), "columnas nuevas solo para 'comercio'.")

### 7. Target / Mean encoding y su riesgo (leakage)

Idea: sustituir cada categoria por la **media del target** (su % de fraude). Compacto y potente.
Pero si calculamos ese % usando **todas** las filas (incluida la que vamos a predecir), el modelo
**ve parte de la respuesta**: memoriza y el score se hincha. Eso es una **fuga de datos**.

> **Overfitting**: el modelo memoriza el entrenamiento en vez de aprender el patron.
> Brilla en train y falla con datos nuevos.

In [ ]:
# Target encoding INGENUO (con todo el train, sin turnos): tiende a sobreajustar.
media_global = train["es_fraude"].mean()
medias_comercio = train.groupby("comercio")["es_fraude"].mean()

train_te_naive = train["comercio"].map(medias_comercio)
test_te_naive = test["comercio"].map(medias_comercio).fillna(media_global)

auc_train_naive = roc_auc_score(train["es_fraude"], train_te_naive)
auc_test_naive = roc_auc_score(test["es_fraude"], test_te_naive.fillna(media_global))
print(f"[Naive] AUC usando solo el encoding de comercio  -> train: {auc_train_naive:.3f} | test: {auc_test_naive:.3f}")
print("El hueco train-test es la señal de alarma del sobreajuste.")

### 8. Hacerlo bien: target encoding por turnos (out-of-fold) + smoothing

- **Out-of-fold**: partimos el train en grupos; calculamos la media con unos grupos y la
  aplicamos al que quedo fuera. Asi una fila nunca usa su propia respuesta.
- **Smoothing**: suavizamos las categorias raras mezclando su media con la media global.
  Formula: `(n·media_categoria + m·media_global) / (n + m)`.

In [ ]:
def target_encode_oof(train_df, test_df, col, target, n_splits=5, m=10, semilla=42):
    media_global = train_df[target].mean()

    # --- TRAIN: por turnos (out-of-fold) ---
    oof = pd.Series(index=train_df.index, dtype=float)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=semilla)
    for idx_tr, idx_val in skf.split(train_df, train_df[target]):
        tr = train_df.iloc[idx_tr]
        stats = tr.groupby(col)[target].agg(["mean", "count"])
        suav = (stats["count"] * stats["mean"] + m * media_global) / (stats["count"] + m)
        val_idx = train_df.index[idx_val]
        oof.loc[val_idx] = train_df.loc[val_idx, col].map(suav).fillna(media_global).values

    # --- TEST: con TODO el train (tambien suavizado) ---
    stats_full = train_df.groupby(col)[target].agg(["mean", "count"])
    suav_full = (stats_full["count"] * stats_full["mean"] + m * media_global) / (stats_full["count"] + m)
    test_enc = test_df[col].map(suav_full).fillna(media_global)
    return oof, test_enc

train_te_oof, test_te_oof = target_encode_oof(train, test, "comercio", "es_fraude", m=20)

auc_train_oof = roc_auc_score(train["es_fraude"], train_te_oof)
auc_test_oof = roc_auc_score(test["es_fraude"], test_te_oof)
print(f"[OOF+smoothing] AUC encoding de comercio -> train: {auc_train_oof:.3f} | test: {auc_test_oof:.3f}")
print("El hueco train-test se reduce: la media es 'honesta' y generaliza mejor.")

### 9. ¿Sube o baja el acierto en datos nuevos?

Montamos la matriz final = One-Hot (baja cardinalidad) + un par de numericas + el `comercio`
codificado, y entrenamos una regresion logistica. Comparamos el target encoding **ingenuo**
frente al **out-of-fold** midiendo el AUC en **test** (datos nuevos).

In [ ]:
num_extra = np.log1p(train[["monto"]].values)            # log del monto como feature numerica
num_extra_test = np.log1p(test[["monto"]].values)

def montar_y_evaluar(encoding_train, encoding_test, etiqueta):
    X_tr = np.hstack([train_ohe, num_extra, encoding_train.values.reshape(-1, 1)])
    X_te = np.hstack([test_ohe, num_extra_test, encoding_test.values.reshape(-1, 1)])
    modelo = LogisticRegression(max_iter=1000)
    modelo.fit(X_tr, train["es_fraude"])
    auc_tr = roc_auc_score(train["es_fraude"], modelo.predict_proba(X_tr)[:, 1])
    auc_te = roc_auc_score(test["es_fraude"], modelo.predict_proba(X_te)[:, 1])
    print(f"{etiqueta:<28} AUC train: {auc_tr:.3f} | AUC test: {auc_te:.3f}")
    return auc_te

print("Modelo completo (One-Hot + log_monto + comercio codificado):")
montar_y_evaluar(train_te_naive.fillna(media_global), test_te_naive, "Target encoding INGENUO")
montar_y_evaluar(train_te_oof, test_te_oof, "Target encoding OUT-OF-FOLD")
print("\nLa version out-of-fold suele generalizar mejor (menor distancia train-test).")

### Reflexion

1. ¿Por que el One-Hot es buena idea para `tipo_tarjeta` pero mala para `comercio`?
2. ¿Que pasa con el AUC de train cuando usamos target encoding ingenuo y por que?
3. ¿Que papel juega el `smoothing` con los comercios que aparecen muy pocas veces?
4. ¿Por que hemos hecho el `train_test_split` ANTES de cualquier codificacion con el target?